In [4]:
# Imports
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Ignore warnings
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

In [5]:
# Checking if we have a GPU
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [24]:
# LOADING
# Train Dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
"berry_dataset/train",
image_size = (224,224),
batch_size = 32
)

# Test Dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    "berry_dataset/test",
    image_size = (224,224),
    batch_size = 32
)

# Validation Dataset 
valid_ds = tf.keras.utils.image_dataset_from_directory(
    "berry_dataset/valid",
    image_size = (224,224),
    batch_size = 32,
    shuffle= False
)

Found 34528 files belonging to 9 classes.
Found 5237 files belonging to 9 classes.
Found 9230 files belonging to 9 classes.


In [29]:
# Printing out the classnames for the `train_ds` dataset. 
class_names = train_ds.class_names
class_names

['bilberry_flower',
 'bilberry_raw',
 'bilberry_ripe',
 'cowberry_bunch_of_flowers',
 'cowberry_bunch_of_raw',
 'cowberry_bunch_of_ripe',
 'cowberry_flower',
 'cowberry_raw',
 'cowberry_ripe']

In [32]:
# Creating an array of lists for `train_ds`
labels = np.concatenate([y.numpy() for _, y in train_ds])
labels

array([5, 6, 1, ..., 7, 1, 8], dtype=int32)

In [33]:
# Checking the shape
labels.shape

(34528,)

In [49]:
# Creating label counts
label_counts_df = pd.DataFrame({"label": labels}).value_counts().sort_index()
label_counts

label
0        2978
1        6505
2        4664
3        3150
4        1126
5        3561
6        5945
7        2835
8        3764
Name: count, dtype: int64

In [48]:
# Creating a more readable label counts
for index, class_name in enumerate(class_names):
    print(f"{class_name}: {label_counts_df[index]}")

bilberry_flower: 2978
bilberry_raw: 6505
bilberry_ripe: 4664
cowberry_bunch_of_flowers: 3150
cowberry_bunch_of_raw: 1126
cowberry_bunch_of_ripe: 3561
cowberry_flower: 5945
cowberry_raw: 2835
cowberry_ripe: 3764


---

#### <strong> Creating the Model </strong>

<br>


In [50]:
# Creating the CNN
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(224,224,3)),
    tf.keras.layers.Rescaling(1./255),

    # FEATURE EXTRACTOR
    # Block 1 
    tf.keras.layers.Conv2D(filters=16, kernel_size = 3, activation = 'relu', padding='same'),
    tf.keras.layers.Conv2D(filters=16, kernel_size = 3, activation = 'relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),

    # Block 2 
    tf.keras.layers.Conv2D(filters=32, kernel_size = 3, activation = 'relu', padding='same'),
    tf.keras.layers.Conv2D(filters=32, kernel_size = 3, activation = 'relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),

    # Block 3 
    tf.keras.layers.Conv2D(filters=64, kernel_size = 3, activation = 'relu', padding='same'),
    tf.keras.layers.Conv2D(filters=64, kernel_size = 3, activation = 'relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),

    # PARAMETERS DICTIONARY
    # kernel_size -- in this case 3, means it's looks in 3x3 block batches
    # filters     -- How many features it creates
    # padding     -- same is YES padding, valid is NO padding (default 1 pixel of padding)
    # activation  -- What metric does it use to reduce filters to 0?
    
    # CLASSIFIER
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(9, activation="softmax")
])

In [51]:
# Compiling
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [52]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 16)   │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 9)              │           585 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 72,665 (283.85 KB)

 Trainable params: 72,665 (283.85 KB)

 Non-trainable params: 0 (0.00 B)